# W5D1 — Classical Text Classification — Guided

**Week 5 · Day 1 · NLP Foundations** · Lab

This morning you computed an IDF with a pencil and watched the word `the` disqualify itself. This
afternoon you do it on twelve thousand real reviews, and you produce **the number the rest of the
week is measured against**.

That is the whole job today. On Friday you fine-tune BERT on this same data, and the only reason
that comparison will mean anything is that today's number was produced honestly: inside a
`Pipeline`, cross-validated on five folds, with the spread reported and the cost recorded.

The last task is the one to remember. You will hand the model two sentences built from **exactly the
same words in a different order**, and it will return exactly the same answer, to fifteen decimal
places, because the two sentences have the same vector. Not a bug — the representation. Everything
in the remaining four days exists to fix it.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ١ — تصنيف النصوص بالطريقة الكلاسيكية

**الأسبوع الخامس · اليوم الأول · أساسيات معالجة اللغة** · معمل

حسبت هذا الصباح تكرارًا مستندًا معاكسًا (IDF) بالقلم، ورأيت كلمة `the` تُسقِط نفسها. وتفعل هذا بعد
الظهر على اثنتي عشرة ألف مراجعة حقيقية، وتُخرج **الرقم الذي يُقاس عليه بقيّة الأسبوع**.

وهذه هي المهمة اليوم كلها. فيوم الجمعة تضبط BERT على البيانات نفسها، والسبب الوحيد الذي يجعل تلك
المقارنة ذات معنى أن رقم اليوم أُنتج بأمانة: داخل `Pipeline`، ومُتحقَّقًا منه تقاطعيًا على خمسة أثلام،
ومعروضًا بتشتّته ومعه كلفته.

والمهمة الأخيرة هي ما يجدر تذكّره. ستُعطي النموذج جملتين مبنيّتين من **الكلمات نفسها بترتيب مختلف**،
فيعيد الجواب نفسه بالضبط إلى خمس عشرة منزلة عشرية، لأن للجملتين المتّجه نفسه. وليس هذا خللًا بل هو
التمثيل نفسه. وكل ما في الأيام الأربعة الباقية موجود ليُصلحه.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Compute an IDF by hand and explain why `TfidfVectorizer` prints a different number for the same
  word without either being wrong.
- Write a preprocessing function, and name a case where it destroyed information you needed.
- Normalise Arabic text, measure what that buys in vocabulary size, and say what it cost.
- Tell a stem from a lemma by looking at the output, and say which one you would deploy.
- Cross-validate a text pipeline with the vectoriser **inside** it, and report mean and spread.
- Read a coefficient list and identify which large weights are corpus artefacts rather than signal.
- Prove, not assert, that TF-IDF cannot represent word order.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب IDF يدويًا وتشرح لماذا تطبع `TfidfVectorizer` رقمًا مختلفًا للكلمة نفسها دون أن يكون
  أحدهما خاطئًا.
- أن تكتب دالة معالجة أولية، وأن تسمّي حالةً أتلفت فيها معلومةً كنت تحتاجها.
- أن تُطبّع النص العربي، وتقيس ما يكسبه ذلك في حجم المعجم، وتقول ما كلّفه.
- أن تميّز الجذع (Stem) من اللَّمَّة (Lemma) بالنظر إلى الخرج، وأن تقول أيّهما تنشر.
- أن تتحقّق تقاطعيًا من خطّ معالجة نصّي والمُتَّجِه (Vectorizer) **داخله**، وأن تعرض المتوسط والتشتّت.
- أن تقرأ قائمة معاملات وتحدّد أي الأوزان الكبيرة أثرٌ للمُدوّنة لا إشارةٌ فيها.
- أن تُبرهن، لا أن تدّعي، أن TF-IDF لا يستطيع تمثيل ترتيب الكلمات.

</div>


## About the data

**Dataset:** `reviews_sentiment` — 12,000 product reviews with a binary sentiment label. One row is
one review: `review_text` (15–600 characters), `sentiment` (0 negative, 1 positive) and `language`.
The classes are exactly balanced, 6,000 each, so 0.5 is the accuracy of guessing and any number you
report is measured against that.

The English reviews come from `fancyzhx/amazon_polarity` (Apache-2.0) and the Arabic ones from
`Ruqiya/Arabic_Reviews_of_SHEIN` (Apache-2.0), sampled and balanced for this course. **This one file
carries four of this week's five labs** — D2 compares embeddings against today's baseline, D4 runs
real token sequences through a transformer block, and D5 fine-tunes BERT on it. Do not swap it.

**The known problem: 480 of the 12,000 reviews — 4% — are in Arabic, not English.** Nothing marks
them in the text, and the `language` column exists so this notebook can check your answer, not so
you can read it. A single-language TF-IDF model treats every Arabic word as a term it saw a handful
of times, and the stretch section measures what that costs on that subset separately. The number is
worse, and by more than the confidence interval.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `reviews_sentiment` — اثنتا عشرة ألف مراجعة منتج مع تسمية مشاعر ثنائية. والصف
الواحد مراجعة واحدة: `review_text` (من ١٥ إلى ٦٠٠ محرف) و`sentiment` (٠ سلبي، ١ إيجابي)
و`language`. والفئتان متوازنتان تمامًا، ستة آلاف لكلٍّ منهما، فتكون ٠٫٥ دقّة التخمين، ويُقاس كل رقم
تعرضه عليها.

المراجعات الإنجليزية من `fancyzhx/amazon_polarity` (رخصة Apache-2.0) والعربية من
`Ruqiya/Arabic_Reviews_of_SHEIN` (رخصة Apache-2.0)، مُنتقاة ومتوازنة لهذه الدورة. **وهذا الملف
الواحد يحمل أربعة من معامل الأسبوع الخمسة**: يقارن اليوم الثاني التمثيلات المتّجهية بأساس اليوم،
ويُمرّر اليوم الرابع متتاليات رموز حقيقية في كتلة محوّل، ويضبط اليوم الخامس BERT عليه. فلا تستبدله.

**والمشكلة المعروفة: ٤٨٠ من الاثنتي عشرة ألفًا — أي ٤٪ — بالعربية لا بالإنجليزية.** ولا يميّزها في
النص شيء، وعمود `language` موجود ليتحقّق هذا الدفتر من جوابك لا لتقرأه منه. ونموذج TF-IDF أحادي
اللغة يعامل كل كلمة عربية كمصطلح رآه مرّاتٍ قليلة، ويقيس قسم التمديد ما يكلّفه ذلك على تلك المجموعة
الجزئية وحدها. والرقم أسوأ، وبفارق يتجاوز فاصل الثقة.

</div>


## Setup

Nothing downloads a model today — TF-IDF and logistic regression are both CPU arithmetic on a
sparse matrix, and the whole lab runs in well under a minute of compute.

One exception: task 2.3 uses NLTK's WordNet lemmatiser, which fetches a 10 MB corpus on first use.
If the classroom Wi-Fi is down the cell says so and falls back to a small hand-written table for the
five words in the exercise, so the comparison still renders.

<div dir="rtl" align="right">

## الإعداد

لا شيء يُنزّل نموذجًا اليوم — فـTF-IDF والانحدار اللوجستي (Logistic Regression) حسابٌ على مصفوفة
مُتفرّقة في المعالج، والمعمل كله يعمل في أقلّ من دقيقة حساب بكثير.

واستثناء واحد: تستخدم المهمة ٢٫٣ لَمّاض WordNet في NLTK، وهو يجلب مُدوّنة بعشرة ميغابايت عند أول
استخدام. وإذا انقطعت شبكة القاعة قالت الخليّة ذلك ورجعت إلى جدول صغير مكتوب باليد للكلمات الخمس في
التمرين، فيظهر الجدول على أي حال.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("scikit-learn", "matplotlib", "nltk")
seed_everything(42)

import hashlib
import json
import re
import time

import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5

# The three documents from this morning, on screen for the whole warm-up.
DOCS = [
    "the service is good",
    "the service is bad",
    "the food is good",
]

reviews = pd.read_parquet(get_dataset("reviews_sentiment"))

print(describe_dataset("reviews_sentiment"))
print(f"\n{len(reviews):,} reviews | classes {reviews.sentiment.value_counts().to_dict()}"
      f" | languages {reviews.language.value_counts().to_dict()}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the same IDF, twice  (≈25 min)

Everything here works. Two implementations of one idea, and they disagree.

First your own: `log₁₀(N / df)`, three lines, on this morning's three documents. It gives `the`
**0.000**, `good` **0.176** and `bad` **0.477** — the slide's numbers.

Then `TfidfVectorizer`, on the same three documents, printing `idf_`: `the` **1.000**, `good`
**1.288**, `bad` **1.693**. Neither is broken. scikit-learn takes a *natural* log, smooths both
counts by 1, and adds 1 at the end, so no term is ever weighted to nothing — its 1.000 is a
**minimum, not an absence**.

What has to survive the change of formula is the **ordering**, and that is what the code checks.
Change a document and watch both tables move together.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: الـIDF نفسه مرّتين (نحو ٢٥ دقيقة)

كل ما هنا يعمل. تنفيذان لفكرة واحدة، وهما يختلفان.

أولًا تنفيذك أنت: `log₁₀(N / df)` في ثلاثة أسطر على مستندات هذا الصباح الثلاثة. فيُعطي `the`
**٠٫٠٠٠** و`good` **٠٫١٧٦** و`bad` **٠٫٤٧٧**، وهي أرقام الشريحة.

ثم `TfidfVectorizer` على المستندات الثلاثة نفسها، طابعًا `idf_`: فـ`the` **١٫٠٠٠** و`good`
**١٫٢٨٨** و`bad` **١٫٦٩٣**. وليس أحدهما مكسورًا. فـscikit-learn تأخذ لوغاريتمًا **طبيعيًا**، وتُملّس
العدّتين بواحد، وتضيف واحدًا في النهاية، فلا يُوزَن مصطلح إلى العدم أبدًا — فـ١٫٠٠٠ عندها **حدٌّ أدنى
لا انعدام**.

والذي يجب أن ينجو من تغيّر الصيغة هو **الترتيب**، وهو ما تفحصه الشيفرة. غيّر مستندًا وراقب
الجدولين يتحرّكان معًا.

</div>


In [ ]:
def idf_by_hand(docs, base=10):
    """log(N / df) per term, over whitespace-tokenised documents."""
    tokenised = [set(d.split()) for d in docs]
    vocabulary = sorted(set().union(*tokenised))
    n = len(docs)
    log = np.log10 if base == 10 else np.log
    return {term: float(log(n / sum(term in doc for doc in tokenised)))
            for term in vocabulary}

hand = idf_by_hand(DOCS)
print("by hand — log10(N / df)")
for term, value in hand.items():
    print(f"  {term:<8} df={sum(term in d.split() for d in DOCS)}  idf={value:.3f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriser = TfidfVectorizer().fit(DOCS)
sklearn_idf = dict(zip(vectoriser.get_feature_names_out(), vectoriser.idf_))

comparison = pd.DataFrame({
    "by_hand_log10": pd.Series(hand),
    "sklearn_idf_": pd.Series(sklearn_idf),
}).sort_values("by_hand_log10")
comparison["hand_rank"] = comparison.by_hand_log10.rank(method="min").astype(int)
comparison["sklearn_rank"] = comparison.sklearn_idf_.rank(method="min").astype(int)

print(comparison.round(3).to_string())
print(f"\nthe:  hand {hand['the']:.3f} vs sklearn {sklearn_idf['the']:.3f}"
      f"  — zero against a floor of one")
print(f"same ordering: {comparison.hand_rank.equals(comparison.sklearn_rank)}")

### Read the two columns before you go on

`the` and `is` are in all three documents. Your formula gives them **0** — they were removed by
arithmetic, and nobody wrote a stopword list. scikit-learn gives them **1.0**, its floor.

`bad` and `food` are each in one document out of three, and they are the largest in **both**
columns. The absolute values are a convention; the ordering is the representation.

<div dir="rtl" align="right">

### اقرأ العمودين قبل أن تُكمل

`the` و`is` في المستندات الثلاثة كلها. وصيغتك تعطيهما **٠** — أزالهما الحساب، ولم يكتب أحدٌ قائمة
كلماتٍ موقوفة. وتعطيهما scikit-learn **١٫٠**، وهو حدّها الأدنى.

و`bad` و`food` كلٌّ منهما في مستند واحد من ثلاثة، وهما الأكبر في **العمودين**. فالقيم المطلقة
اصطلاح، والترتيب هو التمثيل.

</div>


## Section 2 — Core: six tasks  (≈60 min)

1. A preprocessing function, and the case where it destroyed something you needed.
2. Arabic normalisation, with the vocabulary size before and after.
3. Stemming against lemmatisation, five words, side by side.
4. `TfidfVectorizer` + `LogisticRegression` in a `Pipeline`, cross-validated on five folds.
5. The twenty strongest coefficients each way — and the ones that are artefacts.
6. The negation pair, then the order pair that settles it.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. دالة معالجة أولية، والحالة التي أتلفت فيها شيئًا كنت تحتاجه.
٢. تطبيع العربية، ومعه حجم المعجم قبل وبعد.
٣. التجذيع مقابل اللَّمّ، خمس كلمات، جنبًا إلى جنب.
٤. `TfidfVectorizer` مع `LogisticRegression` في `Pipeline`، مُتحقَّقًا منه على خمسة أثلام.
٥. أقوى عشرين معاملًا في كل اتجاه — وأيّها أثرٌ للمُدوّنة.
٦. زوج النفي، ثم زوج الترتيب الذي يحسم المسألة.

</div>


### Task 2.1 — preprocess, then find what it broke

Write the function this morning's pipeline slide described: lower-case, drop punctuation, split on
whitespace. Apply it to ten real reviews and print before and after.

Then go looking for the damage. Somewhere in those ten reviews the function turned something useful
into something useless: a price became two tokens, a decimal split in half, `didn't` became `didn`
and `t`, a model number lost its hyphen. **Find one, print it, and name what was lost.** The
function is not wrong — it is a choice, and this is the price of the choice.

<div dir="rtl" align="right">

### المهمة ٢٫١ — عالِج أوّليًا ثم جِد ما أتلفَته

اكتب الدالة التي وصفتها شريحة مسار المعالجة هذا الصباح: تصغير الحروف، وحذف الترقيم، والتقسيم على
المسافات. طبّقها على عشر مراجعات حقيقية واطبع ما قبل وما بعد.

ثم ابحث عن الضرر. ففي مكانٍ من تلك المراجعات العشر حوّلت الدالة شيئًا نافعًا إلى شيء عديم النفع:
صار سعرٌ رمزين، أو انشقّ عددٌ عشري نصفين، أو صار `didn't` هو `didn` و`t`، أو فقد رقم طراز شرطته.
**جِد واحدة واطبعها وسمِّ ما فُقِد.** والدالة ليست خاطئة بل هي اختيار، وهذا ثمن الاختيار.

</div>


In [ ]:
sample = reviews.sample(10, random_state=SEED)


def preprocess(text):
    """Lower-case, strip punctuation, tokenise on whitespace."""
    # TODO: Lower-case, replace every non-alphanumeric character with a space, then split.
    # مهمة: صغّر الحروف، واستبدل كل محرف غير أبجدي-رقمي بمسافة، ثم قسّم.


# TODO: Print the first 90 characters of each sampled review and its first 12 tokens.
# مهمة: اطبع أول ٩٠ محرفًا من كل مراجعة مُنتقاة وأول ١٢ رمزًا منها.

# TODO: the fragment before and after, and set BROKEN_BY_PREPROCESSING to a one-line note.
# مهمة: `BROKEN_BY_PREPROCESSING` ملاحظةً في سطر واحد.

### Task 2.2 — Arabic normalisation, and what it costs

Ten short Arabic strings are provided. Five pairs, and each pair is **the same sentence written
twice**: once with `أ` and once with `ا`, once with `ة` and once with `ه`, once with the diacritics
and once without, once with a `ــ` stretched letter and once plain.

To a vectoriser those are different terms. Write the normaliser — `أ إ آ` → `ا`, `ة` → `ه`,
`ى` → `ي`, strip the diacritics and the tatweel — and report the vocabulary size before and after.

The drop is large. Then look at what merged: **`على` and `علي` are now the same token.** One is a
preposition, the other is a name. Write the sentence saying what was gained and what was lost.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — تطبيع العربية وما يُكلّفه

عشر سلاسل عربية قصيرة مُعطاة. خمسة أزواج، وكل زوج **الجملة نفسها مكتوبةً مرّتين**: مرّةً بـ`أ`
ومرّةً بـ`ا`، ومرّةً بـ`ة` ومرّةً بـ`ه`، ومرّةً بالتشكيل ومرّةً بدونه، ومرّةً بحرفٍ ممدود بـ`ــ`
ومرّةً بلا مدّ.

وهذه عند المُتَّجِه مصطلحات مختلفة. اكتب المُطبِّع — `أ إ آ` إلى `ا`، و`ة` إلى `ه`، و`ى` إلى `ي`،
وحذف التشكيل والتطويل — واعرض حجم المعجم قبل وبعد.

والهبوط كبير. ثم انظر ما اندمج: **`على` و`علي` صارتا الرمز نفسه.** إحداهما حرف جرّ والأخرى اسم.
اكتب الجملة التي تقول ما كُسِب وما فُقِد.

</div>


In [ ]:
ARABIC_SAMPLES = [
    "الخدمة كانت ممتازة",
    "الخدمه كانت ممتازه",
    "الخِدْمَة كانت مُمْتَازَة",
    "أحببت هذا المنتج كثيرا",
    "احببت هذا المنتج كثيرا",
    "إحببت هذا المنتج كثيراً",
    "المنتج وصل متأخرا على غير المتوقع",
    "المنتج وصل متاخرا على غير المتوقع",
    "الجودة سيئة والسعر مرتفع",
    "الجوده سيئه والسعر مرتفـــع",
]


def normalise_arabic(text):
    """Fold the orthographic variants that carry no meaning here."""
    # TODO: Strip diacritics and tatweel, fold the alef forms, then ة -> ه and ى -> ي.
    # مهمة: احذف التشكيل والتطويل، ووحّد صور الألف، ثم `ة` إلى `ه` و`ى` إلى `ي`.


# TODO: distinct raw tokens that normalisation merged into one.
# مهمة: مختلفين دمجهما التطبيع في واحد.

### Task 2.3 — stem or lemma, on five words

Five words are provided: `studies`, `was`, `feet`, `arrived`, `happiness`. Run NLTK's
`PorterStemmer` and `WordNetLemmatizer` on each and print the three columns side by side.

Look at the stem column before you read on. **Four of the five stems are not English words.**
`studi`, `wa`, `arriv`, `happi`. That is not a bug and not a bad implementation — a stemmer chops
suffixes by rule and never consults a dictionary, so a non-word is the expected output. The
lemmatiser returns `study`, `be`, `foot`, `arrive` — real words, from a real lexicon, at maybe fifty
times the cost per token.

Then say which one you would put in a pipeline, and why. Both answers are defensible.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — جذع أم لَمّة، على خمس كلمات

خمس كلمات مُعطاة: `studies` و`was` و`feet` و`arrived` و`happiness`. شغّل `PorterStemmer`
و`WordNetLemmatizer` من NLTK على كلٍّ منها واطبع الأعمدة الثلاثة جنبًا إلى جنب.

انظر عمود الجذوع قبل أن تُكمل. **أربعة من الجذوع الخمسة ليست كلماتٍ إنجليزية**: `studi` و`wa`
و`arriv` و`happi`. وليس هذا خللًا ولا تنفيذًا سيّئًا — فالمُجذِّع يقصّ اللواحق بقاعدة ولا يُراجع
معجمًا قطّ، فاللاكلمة هي الخرج المتوقّع. أما اللَّمّاض فيعيد `study` و`be` و`foot` و`arrive`، كلماتٍ
حقيقية من معجم حقيقي، بكلفة تبلغ خمسين ضعفًا للرمز ربّما.

ثم قل أيّهما تضع في خطّ معالجة، ولماذا. والجوابان كلاهما قابل للدفاع.

</div>


In [ ]:
WORDS = ["studies", "was", "feet", "arrived", "happiness"]
VERBS = {"was", "arrived"}
FALLBACK_LEMMAS = {"studies": "study", "was": "be", "feet": "foot",
                   "arrived": "arrive", "happiness": "happiness"}

# TODO: wordnet download fails), print the table, and count the stems that are not words.
# مهمة: `wordnet`)، واطبع الجدول، وعُدّ الجذوع التي ليست كلمات.

### Task 2.4 — the baseline, produced honestly

`TfidfVectorizer` then `LogisticRegression`, both **inside a `Pipeline`**, cross-validated with
`StratifiedKFold(5, shuffle=True, random_state=42)`. Report the mean, the standard deviation and
the five individual scores.

The pipeline is not stylistic. IDF is computed **from the corpus**: fit the vectoriser on all 12,000
reviews and then cross-validate, and every fold's training set has already seen the document
frequencies of its own test set. That is the week-2 leak wearing different clothes, it inflates the
score by a fraction of a point, and it is invisible unless you look for it. Inside the `Pipeline`,
`cross_val_score` re-fits the vectoriser on each fold's training rows and the leak cannot happen.

Record the wall-clock too. On Friday you will put this next to a BERT fine-tune, and a comparison
without a cost column is the failure this week exists to prevent.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الأساس مُنتَجًا بأمانة

`TfidfVectorizer` ثم `LogisticRegression`، وكلاهما **داخل `Pipeline`**، مُتحقَّقًا منه تقاطعيًا
بـ`StratifiedKFold(5, shuffle=True, random_state=42)`. اعرض المتوسط والانحراف المعياري والدرجات
الخمس منفردة.

والخطّ ليس أمرًا أسلوبيًا. فالـIDF يُحسَب **من المُدوّنة**: درّب المُتَّجِه على الاثنتي عشرة ألف مراجعة
كلها ثم تحقّق تقاطعيًا، فتكون بيانات تدريب كل ثلم قد رأت أصلًا تكرارات المستندات في بيانات اختباره
هو. وهذا هو تسريب الأسبوع الثاني بثوب آخر، يرفع الدرجة كسرًا من نقطة، ولا يُرى إلا إن بحثت عنه.
وداخل `Pipeline` تُعيد `cross_val_score` تدريب المُتَّجِه على صفوف تدريب كل ثلم، فيستحيل التسريب.

وسجّل الزمن الحقيقي أيضًا. فيوم الجمعة تضع هذا بجانب ضبطٍ لـBERT، والمقارنة بلا عمود كلفة هي الفشل
الذي وُجد هذا الأسبوع لمنعه.

</div>


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

X = reviews.review_text
y = reviews.sentiment

# TODO: shuffled folds, and record the scores, the mean, the std and the elapsed seconds.
# مهمة: الدرجات والمتوسط والانحراف والثواني المنقضية.

# TODO: it takes to score 1,000 reviews — Friday's table has an inference-time column.
# مهمة: فيه عمود لزمن الاستدلال.

### Task 2.5 — read the coefficients, then distrust two of them

Pull the 20 most positive and 20 most negative coefficients out of the fitted model and print them
with their weights.

Most of them are exactly what you would write down yourself: `great`, `excellent`, `waste`,
`terrible`. Those are the model working.

Now find the ones that are not sentiment at all. **`my`, `you`, `off`, `instead`, `nothing`,
`returned`** — words with no polarity that carry large weights because of *how this corpus was
written*, not because of what they mean. Somebody who returned a product writes `returned`; that is
a fact about Amazon reviews in 2013, not about English. **Name one and write the sentence.** A
model whose top features are corpus artefacts will fall over the first time the corpus changes.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — اقرأ المعاملات ثم شكّ في اثنين منها

استخرج أقوى عشرين معاملًا موجبًا وعشرين سالبًا من النموذج المُدرَّب واطبعها مع أوزانها.

أكثرها ما كنت لتكتبه بنفسك: `great` و`excellent` و`waste` و`terrible`. وهذه هي حالة النموذج
عاملًا.

وابحث الآن عمّا ليس من المشاعر أصلًا. **`my` و`you` و`off` و`instead` و`nothing` و`returned`** —
كلمات لا قطبية لها وتحمل أوزانًا كبيرة بسبب **طريقة كتابة هذه المُدوّنة** لا بسبب معناها. فمن أرجع
منتجًا يكتب `returned`، وهذه حقيقة عن مراجعات أمازون سنة ٢٠١٣ لا عن الإنجليزية. **سمِّ واحدة واكتب
الجملة.** فالنموذج الذي أقوى سماته آثارٌ للمُدوّنة يسقط أول مرّة تتغيّر المُدوّنة.

</div>


In [ ]:
# TODO: lists side by side.
# مهمة: خُذ أكبر عشرين معاملًا وأصغر عشرين مع مصطلحاتها واطبع القائمتين جنبًا إلى جنب.

# TODO: CORPUS_ARTEFACT to the sentence explaining it.
# مهمة: الجملة التي تشرحه.

### Task 2.6 — the negation pair, then the pair that settles it

**Part one.** Classify `"the service was good"` and `"the service was not good"`. Print the label
and the probability for both.

You will probably find it gets them **right**. Before you conclude that TF-IDF understands negation,
look at task 2.5's negative list: `not` is sitting near the top of it. The model has learned that
reviews containing the token `not` tend to be negative. That is a correlation over a bag of words,
and it holds until the day someone writes "not a single complaint".

**Part two, and this is the one that settles it.** Classify these two:

```
the food was good but the service was bad
the service was good but the food was bad
```

Same words. Opposite meanings. Now compare their **vectors**, not their predictions:
`(v[0] != v[1]).nnz` is `0` — the two rows of the TF-IDF matrix are identical, bit for bit, so the
two probabilities are equal to the last decimal place. No amount of training data and no choice of
classifier can separate them, because **the difference was destroyed before the model ever saw it**.

Write the sentence naming what a representation would need in order to tell them apart. Then keep
it — Wednesday is that sentence, implemented.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — زوج النفي، ثم الزوج الذي يحسم

**الجزء الأول.** صنّف `"the service was good"` و`"the service was not good"`. واطبع التسمية
والاحتمال لكلٍّ منهما.

وستجد على الأرجح أنه يُصيبهما. وقبل أن تستنتج أن TF-IDF يفهم النفي، انظر إلى القائمة السالبة في
المهمة ٢٫٥: `not` جالسة قريبًا من قمّتها. فقد تعلّم النموذج أن المراجعات التي فيها الرمز `not` تنزع
إلى السلبية. وهذا ارتباط على كيسٍ من الكلمات، يصحّ حتى يوم يكتب أحدهم «not a single complaint».

**والجزء الثاني وهو الحاسم.** صنّف هاتين:

```
the food was good but the service was bad
the service was good but the food was bad
```

الكلمات نفسها. والمعنيان متعاكسان. وقارن الآن **متّجهيهما** لا تنبّؤيهما: فـ`(v[0] != v[1]).nnz`
تساوي `0` — صفّا مصفوفة TF-IDF متطابقان بتًّا ببت، فالاحتمالان متساويان إلى آخر منزلة عشرية. ولا
تستطيع أي كمّية بيانات ولا أي اختيار مُصنّف الفصل بينهما، لأن **الفرق أُتلف قبل أن يراه النموذج**.

اكتب الجملة التي تسمّي ما يحتاجه التمثيل ليميّزهما. ثم احتفظ بها — فالأربعاء هو تلك الجملة مُنفَّذة.

</div>


In [ ]:
NEGATION_PAIR = ["the service was good", "the service was not good"]
ORDER_PAIR = ["the food was good but the service was bad",
              "the service was good but the food was bad"]

# TODO: whether the model got the negated one right.
# مهمة: تنبّأ بجملتَي زوج النفي مع احتمالَيهما، وسجّل هل أصاب النموذج في المنفيّة.

# TODO: the result. Then write ORDER_BLINDNESS: what a representation would need.
# مهمة: `ORDER_BLINDNESS`: ما يحتاجه التمثيل.

## Section 3 — Stretch: two experiments  (≈30 min)

**(a) The stopword list and the word `not`.** scikit-learn's built-in English stopword list contains
`not`, `no` and `never`. Run the pipeline three ways — full stoplist, stoplist minus the negations,
and no stoplist at all — and report all three CV means with their spreads.

Predict the ranking before you run it. Then look at the size of the difference against the standard
deviation you measured in task 2.4, and say whether the experiment settled anything.

**(b) The 4% you cannot read.** Find the non-English reviews with a character-range test — Arabic
lives in `؀`–`ۿ` — then score the model on that subset alone and on the English subset
alone, from the same cross-validated predictions. Do not re-fit; use `cross_val_predict` so each
review is scored by a model that never trained on it.

The gap is several points and it is not noise. Say what you would do about it, in one sentence, if
this were a production model and 4% of your users were writing in that language.

<div dir="rtl" align="right">

## القسم الثالث — التمديد: تجربتان (نحو ٣٠ دقيقة)

**(أ) قائمة الكلمات الموقوفة وكلمة `not`.** تحوي قائمة scikit-learn الإنجليزية المبنيّة `not`
و`no` و`never`. شغّل الخطّ بثلاث طرائق — بالقائمة كاملة، وبالقائمة ناقصةً أدوات النفي، وبلا قائمة
أصلًا — واعرض المتوسطات الثلاثة بتشتّتاتها.

تنبّأ بالترتيب قبل التشغيل. ثم انظر حجم الفرق مقابل الانحراف المعياري الذي قِسته في المهمة ٢٫٤، وقل
هل حسمت التجربة شيئًا.

**(ب) الأربعة بالمئة التي لا تقرأها.** جِد المراجعات غير الإنجليزية باختبار مدى محارف — فالعربية في
`؀`–`ۿ` — ثم قيّم النموذج على تلك المجموعة الجزئية وحدها وعلى الإنجليزية وحدها، من
التنبّؤات المُتحقَّق منها تقاطعيًا نفسها. ولا تُعِد التدريب؛ استخدم `cross_val_predict` ليُصنَّف كل
مراجعة بنموذج لم يتدرّب عليها.

والفجوة عدّة نقاط وليست ضجيجًا. قل ما كنت لتفعله بشأنها، في جملة واحدة، لو كان هذا نموذجًا في
الإنتاج و٤٪ من مستخدميك يكتبون بتلك اللغة.

</div>


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import cross_val_predict

NEGATIONS = {"not", "no", "never", "nor"}

# TODO: negations, and no stoplist — and print the three means with their standard deviations.
# مهمة: واطبع المتوسطات الثلاثة بانحرافاتها المعيارية.

# TODO: cross_val_predict, and report both accuracies and the gap.
# مهمة: واعرض الدقّتين والفجوة.

## Save your artefact

`tfidf_baseline.json` — the accuracy with its spread, the five fold scores, the cost in seconds
both ways, the vocabulary size, the negation and order results, and the language breakdown.

**Friday loads this file.** D5 fine-tunes BERT on the same 12,000 reviews and has to evaluate it on
**the same folds**, so the artefact records the fold specification and a SHA-256 of the concatenated
test indices. If D5's hash does not match this one, the two models were measured on different data
and the comparison is void — which is the most common way this kind of comparison gets published
wrong.

<div dir="rtl" align="right">

## احفظ أثرك

`tfidf_baseline.json` — الدقّة بتشتّتها، ودرجات الأثلام الخمس، والكلفة بالثواني في الاتجاهين، وحجم
المعجم، ونتيجتا النفي والترتيب، وتوزيع اللغات.

**ويُحمّل يوم الجمعة هذا الملف.** فاليوم الخامس يضبط BERT على الاثنتي عشرة ألف مراجعة نفسها وعليه
تقييمه على **الأثلام نفسها**، فيسجّل الأثر مواصفة الأثلام وبصمة SHA-256 لفهارس الاختبار مُتسلسلة.
وإن لم تُطابق بصمة اليوم الخامس هذه فقد قِيس النموذجان على بيانات مختلفة والمقارنة باطلة — وهذه أشيع
طريقةٍ تُنشر بها مقارنة من هذا النوع خاطئةً.

</div>


In [ ]:
fold_test_indices = [test.tolist() for _, test in cv.split(X, y)]
FOLDS_SHA256 = hashlib.sha256(
    json.dumps(fold_test_indices, separators=(",", ":")).encode("utf-8")
).hexdigest()

baseline = {
    "dataset": "reviews_sentiment",
    "n_rows": int(len(reviews)),
    "model": "TfidfVectorizer(min_df=2) + LogisticRegression",
    "vocabulary_size": VOCABULARY_SIZE,
    "cv": {"splitter": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True,
           "random_state": SEED, "folds_sha256": FOLDS_SHA256},
    "accuracy_mean": round(CV_MEAN, 4),
    "accuracy_std": round(CV_STD, 4),
    "accuracy_folds": [round(float(s), 4) for s in CV_SCORES],
    "fit_seconds": round(FIT_SECONDS, 2),
    "inference_seconds_per_1000": round(INFERENCE_SECONDS_PER_1000, 4),
    "negation": NEGATION_RESULT,
    "order_pair": ORDER_RESULT,
    "by_language": {"arabic_rows": DETECTED_ARABIC,
                    "arabic_accuracy": round(ARABIC_ACCURACY, 4),
                    "english_accuracy": round(ENGLISH_ACCURACY, 4)},
    "top_positive": TOP_POSITIVE[:10],
    "top_negative": TOP_NEGATIVE[:10],
}

BASELINE_PATH = ARTEFACT_DIR / "tfidf_baseline.json"
BASELINE_PATH.write_text(json.dumps(baseline, indent=2), encoding="utf-8")

print(json.dumps({k: v for k, v in baseline.items()
                  if k in {"accuracy_mean", "accuracy_std", "fit_seconds",
                           "inference_seconds_per_1000", "vocabulary_size"}}, indent=2))
print(f"\nwrote {BASELINE_PATH.name} — folds {FOLDS_SHA256[:12]}…, "
      f"and Friday has to beat {CV_MEAN:.4f}")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>


In [ ]:
check_close(hand["the"], 0.0,
            "a term in every document must have a hand-computed IDF of exactly 0",
            "يجب أن يكون IDF المحسوب يدويًا لمصطلح في كل المستندات صفرًا تمامًا",
            tol=1e-12)

check_close(sklearn_idf["the"], 1.0,
            "scikit-learn's idf_ for that same term must be 1.0, not 0.0 — its formula has a floor",
            "يجب أن يكون `idf_` في scikit-learn للمصطلح نفسه ١٫٠ لا ٠٫٠ — فلصيغتها حدٌّ أدنى",
            tol=1e-12)

check(comparison.hand_rank.equals(comparison.sklearn_rank),
      "the two IDF formulas must rank the terms identically — if the ordering differs, one of the "
      "two implementations is wrong, and it is not scikit-learn",
      "يجب أن تُرتّب صيغتا IDF المصطلحات ترتيبًا واحدًا — فإن اختلف الترتيب فأحد التنفيذين خاطئ، "
      "وليس هو scikit-learn")

check(VOCAB_AFTER < VOCAB_BEFORE and COLLAPSED_BY_RULE,
      f"Arabic normalisation must strictly reduce the vocabulary AND merge على with علي — "
      f"{VOCAB_BEFORE} terms before, {VOCAB_AFTER} after, and the ى -> ي rule "
      f"{'did' if COLLAPSED_BY_RULE else 'did not'} collapse the pair. Both halves matter: the "
      f"first is what normalisation buys, the second is what it costs",
      f"يجب أن يُقلّص تطبيع العربية المعجم قطعًا وأن يدمج `على` بـ`علي` — {VOCAB_BEFORE} مصطلحًا "
      f"قبل و{VOCAB_AFTER} بعد، وقاعدة `ى` إلى `ي` "
      f"{'دمجت' if COLLAPSED_BY_RULE else 'لم تدمج'} الزوج. والشقّان مهمّان: الأول ما يكسبه "
      f"التطبيع، والثاني ما يُكلّفه")

check([name for name, _ in pipeline.steps] == ["tfidf", "clf"],
      f"the vectoriser must be a step inside the Pipeline, not fitted before the split — the "
      f"steps are {[name for name, _ in pipeline.steps]}, and this is the leakage guard",
      f"يجب أن يكون المُتَّجِه خطوةً داخل `Pipeline` لا مُدرَّبًا قبل التقسيم — والخطوات "
      f"{[name for name, _ in pipeline.steps]}، وهذا حرس التسريب")

check(len(CV_SCORES) == N_SPLITS and CV_STD > 0 and CV_MEAN > 0.75,
      f"cross-validation must produce {N_SPLITS} scores with a non-zero spread and a mean above "
      f"0.75 — got {len(CV_SCORES)} scores, mean {CV_MEAN:.4f}, std {CV_STD:.4f}",
      f"يجب أن يُخرج التحقّق التقاطعي {N_SPLITS} درجات بتشتّت غير صفري ومتوسط فوق ٠٫٧٥ — الناتج "
      f"{len(CV_SCORES)} درجات، والمتوسط {CV_MEAN:.4f}، والانحراف {CV_STD:.4f}")

check(VECTORS_IDENTICAL and order_probabilities[0] == order_probabilities[1],
      "the order pair must produce identical vectors and therefore identical probabilities — if "
      "they differ, the two sentences do not have the same words and the proof is not a proof",
      "يجب أن يُخرج زوج الترتيب متّجهين متطابقين ومن ثمّ احتمالين متطابقين — فإن اختلفا فليست "
      "للجملتين الكلمات نفسها وليس البرهان برهانًا")

check(all(k in baseline for k in ("accuracy_mean", "fit_seconds",
                                  "inference_seconds_per_1000", "negation", "order_pair")),
      f"tfidf_baseline.json must record the cost as well as the accuracy — Friday's table has a "
      f"time column, and the keys present are {sorted(baseline)}",
      f"يجب أن يسجّل `tfidf_baseline.json` الكلفة كما يسجّل الدقّة — فجدول الجمعة فيه عمود زمن، "
      f"والمفاتيح الموجودة {sorted(baseline)}")

report()

## What's next

**W5D2 — Exploring embedding space.** Today's model cannot tell `good` from `excellent`: they are
two unrelated columns, and a review using one of them tells it nothing about the other. Tomorrow
every word becomes a point in a 384-dimensional space where `terrible` sits next to `awful`, and you
measure the distance with the cosine you compute by hand in the first five minutes.

You will also run tomorrow's classifier against today's `tfidf_baseline.json` — and on the negation
pair from task 2.6, where the embedding model gets it right for a better reason than `not` being a
suspicious token.

The order pair, though, survives tomorrow too. It takes Wednesday.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٢ — استكشاف فضاء التمثيل.** لا يستطيع نموذج اليوم أن يميّز `good` من `excellent`:
فهما عمودان لا صلة بينهما، ولا تقول مراجعةٌ استعملت إحداهما شيئًا عن الأخرى. وغدًا تصير كل كلمة
نقطةً في فضاءٍ من ٣٨٤ بُعدًا يجلس فيه `terrible` جوار `awful`، وتقيس المسافة بجيب التمام الذي
تحسبه يدويًا في الدقائق الخمس الأولى.

وستُشغّل مُصنّف الغد أيضًا مقابل `tfidf_baseline.json` اليوم — وعلى زوج النفي من المهمة ٢٫٦، حيث
يُصيبه نموذج التمثيل لسببٍ أفضل من كون `not` رمزًا مشبوهًا.

أما زوج الترتيب فينجو غدًا أيضًا. ويستلزم الأربعاء.

</div>
